# CreditLens — 01 · Exploratory Data Analysis

**You write the code cells; I review.** Each section says *what* to produce and *why*.

EDA answers five questions that shape every later decision:
1. How (im)balanced is the target? → metric + resampling choice
2. Where is the signal? → which features separate defaulters
3. How bad is missingness? → imputation strategy
4. What are the data traps? → e.g. `DAYS_EMPLOYED` anomaly (loader already handles it)
5. What lives in the side tables? → motivates Phase 2 aggregation

## 0 · Setup
Make the `creditlens` package importable from `notebooks/`, import libs, load the data with **our loaders** (`creditlens.data.load`) — don't re-`read_csv` (the loader fixes the `DAYS_EMPLOYED == 365243` anomaly for you).

In [ ]:
# TODO:
#   - sys.path.insert(0, '..')  so 'creditlens' imports from notebooks/
#   - import numpy, pandas, matplotlib.pyplot
#   - from creditlens.data.load import load_application, load_bureau, load_previous_application
#   - from creditlens.config import TARGET, ID_COL
#   - app = load_application();  print app.shape;  app.head()


## 1 · Target balance
`TARGET = 1` = payment difficulty (default-like), `0` = repaid.
Produce: class counts, default rate, a bar plot.
**In a markdown cell, answer:** why do we score with ROC AUC instead of accuracy here?

In [ ]:
# TODO: app[TARGET].value_counts(); app[TARGET].mean(); bar plot of the two classes


## 2 · Missingness
Produce: fraction missing per column, the top ~15 most-missing as a horizontal bar.
Don't drop anything yet — *missing* can itself be predictive.

In [ ]:
# TODO: app.isna().mean().sort_values(ascending=False) -> inspect + plot top 15


## 3 · Where is the signal? EXT_SOURCE_1/2/3
These external scores are the strongest predictors on this dataset.
Produce: correlation of each with `TARGET`, and distribution of each split by target (overlaid histograms).
**Interpret:** what does the sign of the correlation mean?

In [ ]:
# TODO: ext = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']
#       corr with TARGET; per-feature histogram for TARGET==0 vs ==1


## 4 · Engineered ratios beat raw amounts
Raw `AMT_CREDIT` alone is weak — a big loan is fine for a high earner. Ratios capture affordability.
Produce (just preview here, formalized in Phase 2):
`AGE_YEARS = -DAYS_BIRTH/365`, `CREDIT_INCOME_RATIO`, `ANNUITY_INCOME_RATIO`, `CREDIT_TERM = AMT_ANNUITY/AMT_CREDIT`.
Compare their **mean grouped by TARGET**.

In [ ]:
# TODO: build the 4 features on a copy; groupby(TARGET)[cols].mean()


## 5 · Categorical default rates
For a categorical, the useful view is **default rate within each category** (large spread = signal).
Do it for `CODE_GENDER`, `NAME_CONTRACT_TYPE`, `NAME_EDUCATION_TYPE`.

In [ ]:
# TODO: for each col: app.groupby(col)[TARGET].agg(['mean','count']).sort_values('mean')


## 6 · Side tables need aggregation
`bureau` and `previous_application` have many rows per applicant — can't join directly.
Show rows, unique `SK_ID_CURR`, and avg/max rows per applicant. This motivates Phase 2.

In [ ]:
# TODO: load_bureau(), load_previous_application();
#       df.groupby(ID_COL).size() -> count, mean, max per applicant


## Takeaways → Phase 2
Write 3–5 bullet conclusions in your own words: metric choice, top features, ratios to build,
imputation need, and the aggregation plan for the side tables.